In [ ]:
import io
import os
import shutil
import folium
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib import rcParams
import matplotlib.patheffects as path_effects
from matplotlib_scalebar.scalebar import ScaleBar

from shapely.geometry import box, LineString, Point, MultiPoint
from shapely.ops import unary_union, polygonize

from scipy.optimize import curve_fit
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, r2_score, mean_squared_error

import hdbscan
from collections import Counter
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from sklearn.metrics import precision_score, recall_score, f1_score

projected_crs = "EPSG:21037"
DATA_DIR = Path('/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/')

# Clean small data

In [ ]:
df1 = pd.read_csv(DATA_DIR / 'Francis/Garbage_point1.csv')
df2 = pd.read_csv(DATA_DIR / 'Francis/Garbage_point2.csv')

df1 = df1.dropna(how='all')
df2 = df2.dropna(how='all')
df1 = df1.dropna(axis=1, how='all')
df2 = df2.dropna(axis=1, how='all')

df1 = df1.drop('OSM', axis=1) 

In [ ]:
combined_df = pd.concat([df1, df2], ignore_index=True)
combined_df = combined_df.drop_duplicates()
combined_df = combined_df.rename(columns={
    '_Capture_the_location_latitude': 'lat',
    '_Capture_the_location_longitude': 'lon'
})
combined_df['lat'] = pd.to_numeric(combined_df['lat'])
combined_df['lon'] = pd.to_numeric(combined_df['lon'])
combined_df.columns = combined_df.columns.str.lower()

# combined_df

key_columns = [
    'lat', 'lon', 'sub_county', 'ward', 'feature', 
    'garbage/garbage_type', 'garbage/garbage_ownership']
aggregation_rules = {
    col: 'first' for col in combined_df.columns if col not in key_columns + ['comments', 'description']
}

def combine_strings(series):
    # Filter out NA values, convert to string, then join
    non_na_values = series.dropna().astype(str)
    if not non_na_values.empty:
        # Use a distinctive separator, e.g., " | " or " --- "
        return " | ".join(non_na_values.unique())
    return pd.NA # Return pd.NA if all values are NA

aggregation_rules['comments'] = combine_strings
aggregation_rules['description'] = combine_strings

consolidated_df = combined_df.groupby(key_columns, as_index=False).agg(aggregation_rules)
# consolidated_df.to_csv("consolidated_df.csv", index=False)
consolidated_df

In [ ]:
# Function to clean and combine comments and description
def combine_description_comments(row):
    # Handle None, NaN, and equivalent values
    description = row['description']
    comments = row['comments']
    
    # Replace any unwanted values with None
    if pd.isna(description) or description in [None, 'None', 'nan', 'NaN', 'missing', 'balabala']:
        description = None
    if pd.isna(comments) or comments in [None, 'None', 'nan', 'NaN', 'missing', 'balabala']:
        comments = None
    
    # If both are None or NA, return None
    if pd.isna(description) and pd.isna(comments):
        return None
    
    # If both are the same, return one
    if description == comments:
        return description
    
    # If both are different, combine them
    combined = []
    if description:
        combined.append(description)
    if comments:
        combined.append(comments)
    
    return ' | '.join(combined)  # You can choose a different separator if needed

# Apply the function to create a new column for combined text
consolidated_df['combined_description'] = consolidated_df.apply(combine_description_comments, axis=1)


# Function to clean unwanted values across the whole DataFrame
def uniform_none(val):
    if isinstance(val, str):
        # If it's a string like 'None', 'nan', 'NaN', 'missing', 'balabala', 'none', convert it to None
        if val.strip().lower() in ['none', 'nan', 'missing', 'no', 'n\\a', 'na', 'n/s', 'no.', 'nonr']:
            return None
    # If it's already pd.NA or None, leave it as it is
    return val

# Apply the function across the whole DataFrame
consolidated_df = consolidated_df.map(uniform_none)
consolidated_df

# Drop the original comments and description columns if not needed anymore
consolidated_df.drop(['comments', 'description'], axis=1, inplace=True)
consolidated_df.to_csv(DATA_DIR / 'Francis/Cleaned_LA_waste.csv', index=False)

# Display the result
consolidated_df

In [ ]:
len(consolidated_df['ward'].unique())

# Chekc & Plot 

In [ ]:
waste_LA = pd.read_csv(DATA_DIR / 'Francis/withoutbin.csv')
# consolidated_df
waste_LA

## Plot 

In [ ]:
type_counts = consolidated_df["garbage/garbage_type"].value_counts()

# Plot pie chart
plt.figure(figsize=(6, 6))
plt.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%', startangle=140)
plt.title("Distribution of Garbage Types")
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

In [ ]:
consolidated_df[consolidated_df['garbage/garbage_type'] != 'bin'].to_csv(DATA_DIR / 'Francis/withoutbin.csv', index=False)

In [ ]:
# Count occurrences of each garbage type
ownership_counts = consolidated_df["garbage/garbage_ownership"].value_counts()

# Plot pie chart
plt.figure(figsize=(6, 6))
plt.pie(ownership_counts, labels=ownership_counts.index, autopct='%1.1f%%', startangle=140)
plt.title("Distribution of Garbage Ownership")
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

In [ ]:
# 1. Get the center of the map (average of all latitudes and longitudes)
center_lat = consolidated_df['lat'].mean()
center_lon = consolidated_df['lon'].mean()

# 2. Create a basic Folium map
# 'location' sets the initial view, 'zoom_start' sets how zoomed in it is.
m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

# 3. Add a simple marker for each location
for idx, row in consolidated_df.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']] # Just the location for the marker
    ).add_to(m) # Add the marker to our map

# 4. Save the map to an HTML file
map_output_path = 'basic_locations_map.html'
m.save(map_output_path)

print(f"Your basic map has been saved to: {map_output_path}")
print("Just open this HTML file in your web browser to see the points.")

In [ ]:
# --- Basic Scatter Plot using Seaborn ---
plt.figure(figsize=(8, 6)) # Set the size of the plot
sns.scatterplot(data=consolidated_df, x='lon', y='lat')

# Add labels and title for clarity
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Location Points (Scatter Plot - Not a Geographic Map)")
plt.grid(True) # Add a grid for better readability

# Show the plot
plt.show()

# Waste SVI

In [ ]:
Waste_Nairobi_df = pd.read_csv("/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/QGIS/Waste_Nairobi.csv")
Waste_Nairobi_df

# Slum

In [ ]:
slums_gdf = gpd.read_file("/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/Angela/slum_polygon.geojson")
slums_gdf = slums_gdf.to_crs(epsg=21037)
slums_gdf

# Compute Distance to Nearest Slum for Each Waste Point

In [ ]:
# Create GeoDataFrames from latitude and longitude
SVI_gdf = Waste_Nairobi_df.copy()
SVI_gdf['geometry'] = SVI_gdf.apply(lambda row: Point(row['lon'], row['lat']), axis=1)
SVI_gdf = gpd.GeoDataFrame(SVI_gdf, geometry='geometry', crs='EPSG:4326')
SVI_gdf = SVI_gdf.to_crs(epsg=21037)

LA_gdf = waste_LA.copy()
LA_gdf['geometry'] = LA_gdf.apply(lambda row: Point(row['lon'], row['lat']), axis=1)
LA_gdf = gpd.GeoDataFrame(LA_gdf, geometry='geometry', crs='EPSG:4326')
LA_gdf = LA_gdf.to_crs(epsg=21037)


In [ ]:
def evaluate_overlap_reverse(SVI_gdf, LA_gdf, buffer_radius):
    # Buffer the predicted (SVI) points
    SVI_buffered = SVI_gdf.copy()
    SVI_buffered['geometry'] = SVI_buffered.buffer(buffer_radius)

    # Check if each LA point falls within any of the buffered SVI points
    LA_gdf['matched'] = LA_gdf.geometry.apply(lambda point: SVI_buffered.geometry.intersects(point).any())

    # True Positives: LA points that were matched by SVI predictions
    TP = LA_gdf['matched'].sum()
    FN = (~LA_gdf['matched']).sum()

    # Check which SVI predictions matched at least one LA point (to calculate FP)
    SVI_gdf['matched'] = SVI_buffered.geometry.apply(lambda poly: LA_gdf.geometry.intersects(poly).any())
    FP = (~SVI_gdf['matched']).sum()

    precision = TP / (TP + FP) if (TP + FP) else 0
    recall = TP / (TP + FN) if (TP + FN) else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) else 0

    return {
        'Buffer (m)': buffer_radius,
        'TP': TP,
        'FP': FP,
        'FN': FN,
        'Precision': round(precision, 3),
        'Recall': round(recall, 3),
        'F1 Score': round(f1, 3)
    }

results_reverse = evaluate_overlap_reverse(SVI_gdf, LA_gdf, buffer_radius=50)
results_reverse

In [ ]:
def evaluate_overlap(SVI_gdf, LA_gdf, buffer_radius):
    # Create buffer around each local authority (ground truth) point
    LA_buffered = LA_gdf.copy()
    LA_buffered['geometry'] = LA_buffered.buffer(buffer_radius)

    # Check if each SVI point falls within any of the buffered LA points
    SVI_gdf['matched'] = SVI_gdf.geometry.apply(lambda point: LA_buffered.geometry.intersects(point).any())

    # Precision: Of predicted (SVI) points, how many matched
    TP = SVI_gdf['matched'].sum()
    FP = (~SVI_gdf['matched']).sum()

    # Recall: Of ground truth (LA) points, how many were matched by at least one SVI point
    LA_gdf['matched'] = LA_buffered.geometry.apply(lambda poly: SVI_gdf.geometry.intersects(poly).any())
    FN = (~LA_gdf['matched']).sum()

    precision = TP / (TP + FP) if (TP + FP) else 0
    recall = TP / (TP + FN) if (TP + FN) else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) else 0

    return {
        'Buffer (m)': buffer_radius,
        'TP': TP,
        'FP': FP,
        'FN': FN,
        'Precision': round(precision, 3),
        'Recall': round(recall, 3),
        'F1 Score': round(f1, 3)
    }

results = []
for radius in [25, 35, 50, 100, 200, 300, 400, 500]:
    res = evaluate_overlap(SVI_gdf, LA_gdf, radius)
    results.append(res)

results_df = pd.DataFrame(results)
results_df

In [ ]:
# Compute the shortest distance from each waste point to the nearest slum boundary
waste_LA['geometry'] = waste_LA.apply(lambda row: Point(row['lon'], row['lat']), axis=1)
waste_LA = gpd.GeoDataFrame(waste_LA, geometry='geometry', crs='EPSG:4326')
waste_LA = waste_LA.to_crs(epsg=21037)

slums_union = slums_gdf.union_all()

waste_LA['distance_to_slum_m'] = waste_LA.geometry.apply(lambda point: point.distance(slums_union))

# View result
waste_LA[['lat', 'lon', 'distance_to_slum_m']].head()
waste_LA

In [ ]:
# Sort the distances in ascending order
distances = waste_LA['distance_to_slum_m'].sort_values().reset_index(drop=True)

# Compute cumulative percentage
cumulative_percent = np.arange(1, len(distances) + 1) / len(distances) * 100

# Plot
plt.figure(figsize=(10, 6))
plt.plot(distances, cumulative_percent, label='Cumulative % of Waste Points', color='darkblue')

# Add markers or thresholds for interpretation (optional)
plt.axvline(x=250, color='red', linestyle='--', label='100m threshold')
plt.axvline(x=500, color='orange', linestyle='--', label='250m threshold')

# Labels and styling
plt.xlabel('Distance to Nearest Slum (meters)')
plt.ylabel('Cumulative Percentage of Waste Points')
plt.title('Distance Decay Curve: Waste Points Near Slums')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df1 = pd.read_csv('/Users/wenlanzhang/Downloads/VIIRS_NUTS_All_19_21+.csv')
df1